In [1]:
import numpy as np
import tensorflow as tf
from keras import layers, models
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import os
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize

In [2]:
# 1. CARGAR Y FILTRAR DATOS
# (Asumiendo que cargas tu archivo limpio aquí)
ruta_limpia = r'Data\Raw_data\dataset_limpio_y_unificado.npz'
datos = np.load(ruta_limpia)
X_data = datos['X']
Y_data = datos['Y']

# Definir las 8 clases viables (> 600 muestras)
clases_viables = ['N', '/', 'R', 'L', 'V', 'A', 'f', 'F']

# Crear máscara booleana para filtrar
mascara = np.isin(Y_data, clases_viables)
X_filtrado = X_data[mascara]
Y_filtrado = Y_data[mascara]

print(f"Latidos listos para entrenamiento: {len(Y_filtrado)}")

Latidos listos para entrenamiento: 93838


In [3]:
# 2. PREPARAR ETIQUETAS Y TENSORES
# Convertir letras (N, V, R...) a números enteros (0 a 7) para la red neuronal
codificador = LabelEncoder()
Y_numerico = codificador.fit_transform(Y_filtrado)

# Asegurar que el tensor X tenga la dimensión de canal (profundidad = 1 para CNN)
X_cnn = np.expand_dims(X_filtrado, axis=-1) # Pasa de (N, 6, 54) a (N, 6, 54, 1)

# Dividir en datos de Entrenamiento (80%) y Validación (20%)
X_train, X_val, y_train, y_val = train_test_split(X_cnn, Y_numerico, test_size=0.2, random_state=42)

In [4]:
# 3. CALCULAR LOS PESOS DE CLASE (El secreto del balanceo)
# Esto penaliza matemáticamente los errores en las clases más pequeñas (ej. F, f, A)
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
# Convertir el arreglo a un diccionario que Keras pueda entender
diccionario_pesos = dict(enumerate(pesos))
print("\nPesos asignados por clase (La clase N tendrá un peso muy bajo, y la F muy alto):")
for clase, peso in zip(codificador.classes_, pesos):
    print(f"Clase '{clase}': {peso:.4f}")


Pesos asignados por clase (La clase N tendrá un peso muy bajo, y la F muy alto):
Clase '/': 1.6635
Clase 'A': 7.2799
Clase 'F': 17.0304
Clase 'L': 1.9689
Clase 'N': 0.1765
Clase 'R': 1.6856
Clase 'V': 2.8531
Clase 'f': 11.7004


In [5]:
# 4. ARQUITECTURA CNN DESDE CERO
import tensorflow as tf
from tensorflow.keras import layers

num_categorias = 8

inp = layers.Input(shape=(29, 112, 1))

# --- 6 CAPAS CONVOLUCIONALES ---

# Bloque 1
x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp) # Convolucional 1
x = layers.BatchNormalization()(x)
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)   # Convolucional 2
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)

# Bloque 2
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)   # Convolucional 3
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)   # Convolucional 4
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)

# Bloque 3
x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)  # Convolucional 5
x = layers.BatchNormalization()(x)
x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)  # Convolucional 6
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)

# Aplanamos los mapas de características para entrar a las capas densas
x = layers.Flatten()(x)

# --- 6 CAPAS FULLY CONNECTED (DENSAS) ---

# Densa 1
x = layers.Dense(1024, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

# Densa 2
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.4)(x)

# Densa 3
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

# Densa 4
x = layers.Dense(128, activation="relu")(x)

# Densa 5
x = layers.Dense(64, activation="relu")(x)

# Densa 6 (Capa de Salida)
out = layers.Dense(num_categorias, activation="softmax")(x)

modelo = tf.keras.Model(inputs=inp, outputs=out)
modelo.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 29, 112, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 29, 112, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 29, 112, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 112, 32)    │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 29, 112, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 56, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 56, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 14, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 56, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 14, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 28, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 28, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 7, 28, 128)     │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 7, 28, 128)     │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 7, 28, 128)     │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 14, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5376)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     5,506,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 6,496,168 (24.78 MB)

 Trainable params: 6,493,224 (24.77 MB)

 Non-trainable params: 2,944 (11.50 KB)

In [6]:
# 5. COMPILAR Y ENTRENAR
import tensorflow as tf
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Compilamos el modelo
# Usamos solo "accuracy" para evitar el conflicto de dimensiones con sparse_categorical_crossentropy
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Definimos los callbacks para controlar el entrenamiento y guardar el mejor modelo
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6),
    ModelCheckpoint("mejor_modelo.keras", save_best_only=True, monitor="val_accuracy")
]

print("Iniciando entrenamiento de la red convolucional desde cero...")

# Entrenamos el modelo en una sola fase
historial = modelo.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, # Puedes ajustar esto, el EarlyStopping lo detendrá si deja de mejorar
    batch_size=256,
    class_weight=diccionario_pesos,
    callbacks=callbacks,
    verbose=1 # Esto asegura que veas la barra de progreso iterativa
)

Iniciando entrenamiento de la red convolucional desde cero...
Epoch 1/50
229/294 ━━━━━━━━━━━━━━━━━━━━ 2:00 2s/step - accuracy: 0.7239 - loss: 0.5776

KeyboardInterrupt: 

In [ ]:
# 1. HACER PREDICCIONES CON EL SET DE VALIDACIÓN
#modelo=load_model(r'Data\Models\modelo_arritmias_cnn_1.0.keras')
# Obtenemos las probabilidades de la red
y_pred_prob = modelo.predict(X_val)
# Nos quedamos con la clase que tenga mayor probabilidad
y_pred_clases = np.argmax(y_pred_prob, axis=1)

# Nombres de las clases (recuperados de tu codificador anterior)
etiquetas = codificador.classes_

# 2. MATRIZ DE CONFUSIÓN
matriz = confusion_matrix(y_val, y_pred_clases)

plt.figure(figsize=(10, 8))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', 
            xticklabels=etiquetas, yticklabels=etiquetas)
plt.title('Matriz de Confusión: ¿Qué está confundiendo la red?')
plt.ylabel('Latido Real (Verdad)')
plt.xlabel('Predicción del Modelo')
plt.show()

# 3. REPORTE DE CLASIFICACIÓN (Métricas duras)
print("\n=== REPORTE DE CLASIFICACIÓN (PRECISIÓN POR CLASE) ===")
print(classification_report(y_val, y_pred_clases, target_names=etiquetas))

# 4. CURVA ROC MULTICLASE
# Necesitamos binarizar las etiquetas para hacer ROC por cada clase
y_val_bin = label_binarize(y_val, classes=np.arange(len(etiquetas)))

plt.figure(figsize=(10, 8))
for i in range(len(etiquetas)):
    # Calcular Tasa de Falsos Positivos y Verdaderos Positivos
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr) # Área bajo la curva
    
    plt.plot(fpr, tpr, lw=2, label=f'Clase {etiquetas[i]} (AUC = {roc_auc:.2f})')

# Línea diagonal (el modelo adivina al azar)
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC: Capacidad de distinción por arritmia')
plt.legend(loc="lower right")
plt.show()

In [ ]:
archivo_modelo=r"Data\Models\modelo_arritmias_cnn_4.0"

# Guardar el modelo físico en tu disco duro
print(f"\nGuardando el modelo definitivo en: {archivo_modelo}")
modelo.save(archivo_modelo)
print("-" * 50)
print("¡Modelo entrenado y guardado con éxito! Listo para producción.")
